In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import data_tools as data
import xgboost as xgb
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.compose import ColumnTransformer
import osmnx as ox 

In [ ]:
df_zones = gpd.read_file("/Users/sai-z/Desktop/nyc-taxi-demand-forecasting/taxi_zones/taxi_zones.shp")

In [ ]:
df_zones.plot()

In [ ]:
tags = {
    "amenity": "restaurant",
    "tourism": "hotel"
}

ox.settings.overpass_settings = (
    '[out:json][timeout:180][date:"2023-12-31T23:59:59Z"]'
)

ox.settings.requests_timeout = 180
ox.settings.use_cache = True
ox.settings.log_console = True


zones_ll = df_zones.to_crs("EPSG:4326") # Convert the zones into coordinates

In [ ]:
nyc_polygon = zones_ll.geometry.union_all()
poi_nyc = ox.features_from_polygon(
    nyc_polygon,
    tags=tags
)

In [ ]:
poi_nyc.to_parquet("osm_restaurants_hotels_2023.parquet")

In [ ]:
pois = poi_nyc.reset_index()
pois["category"] = "restaurant"

pois.loc[
    pois["tourism"] == "hotel",
    "category"
] = "hotel"
poi_points = pois.copy()
poi_points["geometry"] = (
    poi_points.geometry.representative_point()
)
assigned = gpd.sjoin(
    poi_points,
    zones_ll[["LocationID", "geometry"]],
    how="inner",
    predicate="within"
)

In [ ]:
counts = (
    assigned.groupby(["LocationID", "category"])
    .size()
    .unstack(fill_value=0)
) # count restaurant and hotels per zone

In [ ]:
zone_ids = pd.Index(
    zones_ll["LocationID"],
    name="LocationID"
)

counts = counts.reindex(
    index=zone_ids,
    columns=["restaurant", "hotel"],
    fill_value=0
)
counts = counts.rename(
    columns={
        "restaurant": "restaurant_count",
        "hotel": "hotel_count"
    }
).reset_index()

In [ ]:
counts.to_parquet("zone_poi_counts_2023.parquet", index=False)